In [ ]:
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
# from scipy import stats

In [ ]:
import pickle
import os

In [ ]:
os.makedirs('pkl', exist_ok=True)

times = np.arange(4500, 11701, 30)
# filelist = [f"OUT_3D/GOAMAZON_2pulse_1024_zapped_{t:010d}.nc" for t in times]

In [ ]:
for t in times:
    sam = xr.open_dataset(f"OUT_3D/GOAMAZON_2pulse_1024_zapped_{t:010d}.nc")
    qt = sam.QV + sam.QN
    wi = sam.W
    w = (wi.shift(z=-1, fill_value=0.0) + wi) * 0.5
    qcl = sam.QN
    tr = sam.TR01
    tv = sam.TABS * (1.0 + 0.608 * sam.QV * 0.001 - sam.QN * 0.001)
    wm = w.mean(axis=(2, 3))
    ws = w.std(axis=(2, 3))
    trm = tr.mean(axis=(2, 3))
    trs = tr.std(axis=(2, 3))
    qtm = qt.mean(axis=(2, 3))
    qts = qt.std(axis=(2, 3))
    tvm = tv.mean(axis=(2, 3))
    qclm = qcl.mean(axis=(2, 3))
    dict = {
        "wm": wm,
        "ws": ws,
        "trm": trm,
        "trs": trs,
        "qtm": qtm,
        "qts": qts,
        "tvm": tvm,
        "qclm": qclm
    }
    print(f"Writing out time step {t:010d} ...")
    for key, value in dict.items():
        with open(f"pkl/{key}_{t:010d}.pkl", "wb") as f:
            pickle.dump(value, f)
    sam.close()
    del qt, wi, w, qcl, tr, tv

In [ ]:
# Combine per-timestep pickle files into time x height arrays
keys = ["wm", "ws", "trm", "trs", "qtm", "qts", "tvm", "qclm"]

combined = {}
for key in keys:
    slices = []
    valid_times = []
    for t in times:
        fn = f"pkl/{key}_{t:010d}.pkl"
        if not os.path.exists(fn):
            continue
        with open(fn, "rb") as f:
            arr = pickle.load(f)

        # convert xarray -> numpy if needed, then squeeze singleton dims
        if hasattr(arr, "values"):
            arr = arr.values
        arr = np.asarray(arr).squeeze()  # expected: (z,)
        slices.append(arr)
        valid_times.append(t)

    if len(slices) == 0:
        print(f"No files found for {key}")
        continue

    combined[key] = np.stack(slices, axis=0)  # (time, height)
    print(f"{key}: {combined[key].shape} (time, height), n_times={len(valid_times)}")

# Optional: save combined arrays
for key, arr in combined.items():
    with open(f"pkl/{key}.pkl", "wb") as f:
        pickle.dump(arr, f)